# 5.5 阶段三：buffer 复用（开 pingpong）

BN1 分核解决了「核不够用」，但每个核内部还是**串行**的：搬完一块 KV 才能算，算完才能搬下一块——搬运单元和计算单元轮流空等。

阶段三引入**乒乓（pingpong）双缓冲**。

## 1. 原理

把 KV 的搬运缓冲分成 A/B 两份，交替使用：

```text
时刻0：  搬 K0/V0 → bufferA
时刻1：  算 bufferA(K0/V0)   同时   搬 K1/V1 → bufferB
时刻2：  算 bufferB(K1/V1)   同时   搬 K2/V2 → bufferA
   ...（乒乓往复）
```

任何时刻**一份数据在被计算、另一份在被填充**——搬运时间被计算时间「藏」了起来。

## 2. buffer 布局：Q 常驻 + KV pingpong

为什么 Q 不乒乓？Q 是**驻留片上**的（整个 S2 循环反复使用），只需要在循环前搬一次；K、V 才是逐块流进的——所以标准布局是：

| 张量 | buffer 策略 | 搬运次数 |
|--|--|--|
| Q | 单 buffer 常驻 | 1 次（循环外） |
| K | pingpong × 2 | ceil(S2/s2Block) 次 |
| V | pingpong × 2 | 同 K |

## 3. 代码变化

```cpp
// Ascend C 的 TQue 天然支持双缓冲：BUFFER_NUM = 2
TPipe pipe;
TQue<TPosition::VECIN, MixQuePosition::A1> queK, queV;
pipe.InitBuffer(queK, 2, sizeof(half) * s2Block * dSize);   // 2 = pingpong
pipe.InitBuffer(queV, 2, sizeof(half) * s2Block * dSize);

// 循环外：Q 一次性搬入并常驻
LocalTensor<half> q = queQ.DeQue<half>();

// 循环内：DeQue 自动在 A/B 两份间轮转
for (int s2Idx = 0; s2Idx < s2LoopTimes; s2Idx++) {
    // EnQue 隐含生产者 SetFlag、DeQue 隐含消费者 WaitFlag
    // （第 1 章的 TQue 同步语义在这里生效）
    CopyInKV(s2Idx);                    // 后台预取下一块
    LocalTensor<half> k = queK.DeQue<half>();
    LocalTensor<half> v = queV.DeQue<half>();
    // 四拍循环（C1 → V1 → C2 → V2）作用于当前块
    ...
    queK.FreeTensor(k);                 // 释放，允许搬运单元写入另一份
    queV.FreeTensor(v);
}
```

> 第 1 章讲过的 **TQue 队列同步**（EnQue=SetFlag、DeQue=WaitFlag）就是乒乓的实现机制——`BUFFER_NUM=2` 后框架自动安排交替，不需要手写 flag。

## 4. 收益与新的瓶颈

| 指标 | 阶段二（串行） | 阶段三（pingpong） |
|--|--|--|
| 搬运与计算 | 严格串行交替 | 时间上重叠 |
| 单核耗时 | 搬运 + 计算之和 | max(搬运, 计算) |

新的瓶颈：**计算内部仍串行**。四拍之间 C1→V1→C2→V2 严格先后（数据依赖），Cube 与 Vector 互相等待——这是阶段四 preload 要解决的问题。

## 小测验

1. Q 为什么用单 buffer 常驻而不是乒乓？
2. `BUFFER_NUM=2` 后，`FreeTensor` 释放的那份缓冲立刻可以被谁写入？
3. 乒乓之后，单核耗时从「搬运+计算」变成什么？新的浪费在哪里？